# Anatomía de una Falla · Parte II — 04 · Modelo final (CatBoost con CV+, ACI de flota y política de inspección)

Este es el **modelo final** del proyecto y el que respalda el abstract del congreso (`paper_phm2027/main.tex`).
Reemplaza a la etapa 1 (notebook `02`, con 11 motores de calibración).

**Cómo regenerarlo:** `python scripts/run_all.py --workers 1 --only cv_fit cv_eval entrega`. Este notebook sólo lee
`outputs/full/cvplus*.json` y `cvplus_intervals.parquet`: no necesita los datos crudos.

## Metodología final

1. **Modelo de motor sano y residuos.** Una red chica predice cada sensor desde altitud, Mach, potencia y temperatura de
   entrada, entrenada sólo con vuelos sanos. Residuo = medido − esperado.
2. **Variables que sólo miran hacia atrás:** residuo menos la línea base del motor, medias y pendientes móviles (5 y 10
   vuelos), número de vuelo, clase y condiciones del vuelo.
3. **CatBoost multicuantil (5 / 50 / 95 %)** con validación cruzada **por motor**: los 60 motores de desarrollo se
   dividen en 5 grupos estratificados por subconjunto. Para cada grupo se reentrena **todo** (modelo sano, residuos y
   CatBoost) sin esos motores, y se predice sobre ellos. Así los 60 motores aportan errores fuera de muestra para
   calibrar (antes eran 11). Las iteraciones (200, comunes a los grupos) se eligen por pérdida pinball en desarrollo.
4. **Intervalos CV+ asimétricos.** De un 10 % de error permitido, 3 % para sobreestimar la vida (la cola peligrosa) y
   7 % para subestimarla.
5. **ACI de flota (variante online).** Los motores de prueba se recorren en un orden de llegada; cuando cada uno falla y
   se conoce su vida real, el nivel de cada cola se corrige según cuántas veces su historia quedó fuera del intervalo.
   Se promedia sobre 30 órdenes aleatorios.
6. **Política de inspección.** Se ordena inspeccionar en el primer vuelo en que la cota inferior CV+ (sin ACI) cae por
   debajo de un umbral H. Si ejecutarla lleva τ = 5 vuelos (supuesto), es tardía cuando quedan menos. H se elige en
   desarrollo para no superar 5 % de tardías.

Todas las reglas (método, iteraciones, umbral) se eligen con motores de desarrollo; los 39 motores de prueba oficiales
se usan para evaluar. Los intervalos de confianza remuestrean **motores** (5000 réplicas).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from src.ncmapss import nb
MODE = "full"          # "sample" si corriste el pipeline con --sample

In [2]:
import json
import pandas as pd
import plotly.graph_objects as go
from src.ncmapss.report import entrega as E
pd.set_option('display.max_colwidth', None)
R = nb.load("cvplus", MODE)
DEC = nb.load("cvplus_decision", MODE)
FIT = nb.load("cvplus_fit", MODE)
print("Motores de desarrollo:", R["units_dev"], "· de prueba:", R["units_test"], "· método elegido:", R["chosen"])

Motores de desarrollo: 60 · de prueba: 39 · método elegido: cvplus


## 1 · Decisión en desarrollo

La regla se fijó antes de mirar la prueba: tolerancia a sobreestimar, cercanía a la cobertura objetivo y menor ancho.
CV+ global cumple la tolerancia de 3 % y queda en 90 % de cobertura; la variante por tramos se descarta.

In [3]:
cols = {"coverage": "cobertura", "miss_over": "sobreestima", "miss_under": "subestima", "width": "ancho [vuelos]", "rmse": "RMSE"}
pd.DataFrame(DEC["desarrollo"]).T[list(cols)].rename(columns=cols).round(3)

,cobertura,sobreestima,subestima,ancho [vuelos],RMSE
cvplus,0.900,0.030,0.070,24.548,8.532
cvplus_tramos,0.892,0.037,0.071,25.092,8.532


## 2 · Cuántas iteraciones

Pérdida pinball fuera de grupo (sólo motores de desarrollo) según el número de iteraciones de CatBoost.

In [4]:
c = FIT["curva_pinball"]
fig = go.Figure(go.Scatter(x=c["iter"], y=c["loss"], mode="lines+markers"))
fig.add_vline(x=FIT["iteraciones"], line_dash="dot", annotation_text=f"elegidas: {FIT['iteraciones']}")
fig.update_layout(height=320, xaxis_title="Iteraciones", yaxis_title="Pinball fuera de grupo", template="simple_white")
fig.show()

## 3 · Resultados en los 39 motores de prueba

- **TFT / CatBoost · 11 motores:** etapa 1 (notebook `02`), calibrados con 11 motores (CQR por tramos, sin ACI).
- **CatBoost CV+ · 60 motores:** el modelo final, sin actualización online.
- **CatBoost CV+ y ACI:** el modelo final con la actualización de flota.

Para comparar con la etapa 1 en igualdad de condiciones: sin ACI, la línea base es «CatBoost · 11 motores»; con ACI,
la línea base con ACI está en `results.json` (sobreestimación 4,7 %, cobertura 88,2 %, ancho 18,8 vuelos).

In [5]:
rows = []
for m, name in E.METHODS.items():
    t, ci = R["test"][m], R["ci"][m]
    rows.append({"método": name, "RMSE": t["rmse"], "score NASA": t["nasa"],
                 "cobertura [%]": 100 * t["coverage"], "IC cobertura": f"{100*ci['cov'][0]:.1f} – {100*ci['cov'][1]:.1f}",
                 "sobreestima [%]": 100 * t["miss_over"], "IC sobreestima": f"{100*ci['over'][0]:.1f} – {100*ci['over'][1]:.1f}",
                 "ancho [vuelos]": t["width"]})
res = nb.load("results", MODE)
b = next(r for r in res["metrics_table"] if r["modelo"] == "CatBoost" and r["intervalo"].startswith("ACI"))
rows.insert(2, {"método": "CatBoost · 11 motores y ACI", "RMSE": b["RMSE"], "score NASA": b["score NASA"],
                "cobertura [%]": b["cobertura [%]"], "IC cobertura": "—", "sobreestima [%]": b["% real < cota inferior"],
                "IC sobreestima": "—", "ancho [vuelos]": b["ancho medio [vuelos]"]})
pd.DataFrame(rows).round(2)

,método,RMSE,score NASA,cobertura [%],IC cobertura,sobreestima [%],IC sobreestima,ancho [vuelos]
0,TFT · 11 motores,7.48,0.85,85.37,79.2 – 90.7,6.62,3.6 – 10.4,19.58
1,CatBoost · 11 motores,6.51,0.65,87.10,80.8 – 92.3,6.97,3.6 – 10.9,18.19
2,CatBoost · 11 motores y ACI,6.51,0.65,88.20,—,4.70,—,18.80
3,CatBoost CV+ · 60 motores,6.61,0.70,95.76,91.0 – 98.9,1.14,0.1 – 2.5,25.79
4,CatBoost CV+ y ACI,6.61,0.70,93.49,88.7 – 97.6,1.87,0.3 – 3.3,23.89


### Diferencias pareadas (IC 95 %, remuestreando motores)

Si el intervalo no cruza el cero, la diferencia es clara con 39 motores.

In [6]:
diff = []
for k, v in R["ci"].items():
    if " - " in k:
        for metric, d in v.items():
            diff.append({"comparación": k, "métrica": metric, "IC 95 % de la diferencia": f"{d['dif_ic95'][0]:.3f} a {d['dif_ic95'][1]:.3f}"})
pd.DataFrame(diff)

,comparación,métrica,IC 95 % de la diferencia
0,cvplus - base_catboost,rmse,-0.352 a 0.557
1,cvplus - base_catboost,ae,-0.158 a 0.539
2,cvplus - base_catboost,nasa,-0.011 a 0.128
3,cvplus - base_catboost,cov,0.056 a 0.119
4,cvplus - base_catboost,over,-0.091 a -0.030
5,cvplus - base_catboost,width,7.172 a 8.025
6,base_tft - base_catboost,rmse,0.358 a 1.561
7,base_tft - base_catboost,ae,0.402 a 1.282
8,base_tft - base_catboost,nasa,0.094 a 0.324
9,base_tft - base_catboost,cov,-0.053 a 0.015


In [7]:
go.Figure(E._ci_fig(R, "coverage", "cov", R["target"], "objetivo 90 %", "Cobertura en prueba [%]", [70, 100])).show()
go.Figure(E._ci_fig(R, "miss_over", "over", R["alpha_over"], "tolerancia 3 %", "Sobreestimación en prueba [%]", [0, 12])).show()

## 4 · Por subconjunto, clase de vuelo y tramo de vida remanente (CV+)

In [8]:
cols = {"rmse": "RMSE", "coverage": "cobertura", "miss_over": "sobreestima", "width": "ancho", "n": "vuelos"}
display(pd.DataFrame(R["per_ds"]).T[list(cols)].rename(columns=cols).round(3))
display(pd.DataFrame(R["per_fc"]).T[list(cols)].rename(columns=cols).rename(index={"1": "corto", "2": "mediano", "3": "largo"}).round(3))
pd.DataFrame(R["buckets"]).set_index("tramo")[list(cols)].rename(columns=cols).round(3)

,RMSE,cobertura,sobreestima,ancho,vuelos
DS01,6.050,0.997,0.000,25.958,337.0
DS02,6.488,1.000,0.000,24.948,199.0
DS03,6.576,0.956,0.000,24.281,432.0
DS04,8.476,0.979,0.009,31.128,340.0
DS05,5.219,1.000,0.000,25.157,323.0
DS06,4.242,0.984,0.000,24.447,318.0
DS07,8.949,0.821,0.000,26.048,340.0
DS08a,5.604,0.944,0.056,25.133,377.0
DS08c,6.360,0.961,0.039,24.699,233.0


,RMSE,cobertura,sobreestima,ancho,vuelos
corto,5.757,0.980,0.008,26.401,1068.0
mediano,6.764,0.984,0.012,26.494,1008.0
largo,7.421,0.896,0.015,24.143,823.0


,RMSE,cobertura,sobreestima,ancho,vuelos
tramo,,,,,
0–9,3.597,0.954,0.044,11.031,390
10–24,3.387,0.968,0.014,16.328,585
25–49,7.135,0.954,0.008,26.580,975
50–74,6.958,0.966,0.000,36.506,754
75–999,12.181,0.923,0.000,38.343,195


## 5 · Motor a motor

Cada predicción usa sólo la historia hasta ese vuelo. Cambiá `MOTOR` para ver otro motor de prueba.

In [9]:
it = pd.read_parquet(nb.output_path("cvplus_intervals.parquet", MODE))
MOTOR = "DS02_u11"
g = it[it.uid == MOTOR].sort_values("cycle")
fig = go.Figure([
    go.Scatter(x=g.cycle, y=g.hi, mode="lines", line_width=0, showlegend=False, hoverinfo="skip"),
    go.Scatter(x=g.cycle, y=g.lo, mode="lines", fill="tonexty", name="intervalo CV+ (cota inferior abajo)"),
    go.Scatter(x=g.cycle, y=g.q50, mode="lines", name="predicción (mediana)"),
    go.Scatter(x=g.cycle, y=g.RUL, mode="lines", line_dash="dash", name="RUL real")])
fig.update_layout(height=380, xaxis_title="Número de vuelo", yaxis_title="Vuelos restantes", template="simple_white")
fig.show()

## 6 · De la predicción a la inspección

Vida útil desperdiciada = vuelos que quedaban cuando se inspeccionó, menos τ. Se compara contra una vida fija por clase
de vuelo, elegida también en desarrollo con la misma regla de tardías. La política con la mediana se muestra porque
rinde parecido: la ganancia frente a la vida fija viene sobre todo de tener un pronóstico.

In [10]:
rows = []
for tau, pol in R["inspection"]["by_tau"].items():
    for name in ("cota inferior", "mediana", "vida fija por clase"):
        r = pol[name]
        rows.append({"τ [vuelos]": int(tau), "política": name, "umbral H": r.get("H"), "tardías": f"{r['late_engines']} de {R['units_test']}",
                     "vida desperdiciada": round(r["waste"], 2), "IC 95 %": f"{r['ci']['waste'][0]:.1f} – {r['ci']['waste'][1]:.1f}"})
display(pd.DataFrame(rows))
go.Figure(E._policy_fig(R)).show()

,τ [vuelos],política,umbral H,tardías,vida desperdiciada,IC 95 %
0,3,cota inferior,4.0,2 de 39,8.08,6.9 – 9.2
1,3,mediana,13.0,2 de 39,9.23,8.2 – 10.1
2,3,vida fija por clase,NaN,1 de 39,15.90,12.4 – 19.6
3,5,cota inferior,6.0,1 de 39,8.92,7.7 – 10.1
4,5,mediana,15.0,1 de 39,9.18,8.2 – 10.1
5,5,vida fija por clase,NaN,1 de 39,15.90,12.4 – 19.6
6,10,cota inferior,8.0,2 de 39,6.62,5.4 – 7.9
7,10,mediana,16.0,3 de 39,5.36,4.6 – 6.2
8,10,vida fija por clase,NaN,1 de 39,15.90,12.4 – 19.6


## 7 · Límites que hay que decir

In [11]:
v1 = nb.load("cvplus_v1_early_stopping", MODE)
t, a, dev = R["test"]["cvplus"], R["test"]["cvplus_aci"], R["development"][R["chosen"]]
for s in [
    f"<b>La prueba se miró dos veces.</b> La primera versión (early stopping por grupo) dio score NASA {v1['test']['cvplus']['nasa']:.3f}; "
    f"después se pasó a {FIT['iteraciones']} iteraciones comunes elegidas sólo con pinball en desarrollo ({t['nasa']:.3f}). La regla no usa la "
    "prueba, pero la decisión de rediseñar vino después de verla.",
    f"<b>Sobrecobertura en prueba.</b> CV+ da {100*dev['coverage']:.1f} % en desarrollo y {100*t['coverage']:.1f} % en prueba: los motores de "
    f"prueba son más fáciles (RMSE {t['rmse']:.1f} contra {dev['rmse']:.1f}). ACI la reduce a {100*a['coverage']:.1f} %, a costa de subir la "
    f"sobreestimación de {100*t['miss_over']:.1f} % a {100*a['miss_over']:.1f} %.",
    "<b>ACI usa motores de prueba.</b> Es una actualización online: incorpora la vida real de cada motor de prueba cuando falla. "
    "No es una evaluación offline pura.",
    f"<b>Son {R['units_test']} motores.</b> Cada inspección tardía vale {100/R['units_test']:.1f} puntos; τ = 5 es un supuesto, no un dato de "
    "mantenimiento real. Los motores y su degradación son simulados; los perfiles de vuelo son reales.",
]:
    nb.html("<p>" + s + "</p>")